# ETIQUETADOR
### Informe del proyecto
#### Desarrollo y análisis de eficiencia

---

## Índice

1. [Introducción](#1.-Introducción)
2. [Descripción del proyecto](#2.-Descripción-del-proyecto)
3. [Desarrollo](#3.-Desarrollo)
4. [Análisis de parámetros](#4.-Análisis-de-parámetros)
   - 4.1 Valor de K en KNN
   - 4.2 Número de colores en K-Means
   - 4.3 Inicialización de centroides
   - 4.4 Características usadas en KNN
5. [Análisis de eficiencia](#5.-Análisis-de-eficiencia)
6. [Resultados globales del sistema](#6.-Resultados-globales-del-sistema)
7. [Conclusiones](#7.-Conclusiones)

---


## 1. Introducción

En este proyecto hemos desarrollado un sistema sencillo capaz de etiquetar imágenes de prendas de ropa de forma automática. El programa identifica el tipo de prenda y sus colores principales, permitiendo búsquedas como "red shirt" o "blue jeans" en una tienda online.

La aplicación más obvia es una tienda online: con un etiquetado así, el usuario podría buscar directamente cosas como "red shirt", "black sandals" o "pink dress". Así, el sistema se encarga de revisar todos los productos automáticamente.

Para resolver el problema nos hemos apoyado en dos algoritmos bastante clásicos: **KNN**, que se encarga de reconocer la forma de la prenda (aprendizaje supervisado), y **K-Means**, que detecta los colores dominantes (agrupamiento no supervisado). El sistema trabaja con **8 categorías de prendas** y **11 colores** posibles.

## 2. Descripción del proyecto

El proyecto está organizado en tres bloques que funcionan de forma encadenada: primero clasificamos la forma de la prenda, luego detectamos sus colores y finalmente ofrecemos un buscador que combina ambas etiquetas.

### Clasificación de prendas con KNN

1. Se suben las imágenes de entrenamiento y de prueba (train y test).
2. Se extraen **shape features**: máscara de silueta 32×32, imagen en gris 32×32, proyecciones de filas/columnas y descriptores de bounding-box (aspect ratio, fill ratio, etc.).
3. El KNN compara la imagen nueva con las de entrenamiento usando distancia euclídea.
4. Asigna la clase según el vecino más cercano (K=1, elegido automáticamente).

Las **8 clases** posibles son: Dresses, Shirts, Flip Flops, Shorts, Jeans, Socks, Sandals y Handbags.

### Detección de colores con K-Means

1. Se aplica `garment_mask` para extraer solo los píxeles de la prenda (sin fondo ni piel).
2. K-Means agrupa esos píxeles en clusters de color con inicialización kmeans++.
3. `find_bestK` determina automáticamente el número óptimo de clusters (umbral 20%).
4. Cada centroide RGB se traduce a un nombre de color mediante `get_color_prob()` de `utils.py`.

Los **11 colores** posibles son: Red, Green, Black, Orange, Blue, Grey, Brown, Purple, White, Yellow, Pink.

### Búsqueda de productos

Cuando ya tenemos cada imagen etiquetada con una forma y un color, la búsqueda combinada es casi inmediata. Si alguien busca "blue jeans", el sistema filtra las imágenes cuyo color principal es Blue y cuya forma es Jeans.

## 3. Desarrollo

El código está separado en distintos archivos para que sea más fácil de entender y modificar:

| Archivo | Función principal |
|---|---|
| `KNN.py` | KNN con desempate por distancia y búsqueda vectorizada |
| `Kmeans.py` | K-Means con kmeans++, WCD euclídea y find_bestK adaptativo |
| `shape_features.py` | Extracción de silueta: foreground_mask, garment_mask, skin_mask, extract_shape_features |
| `utils.py` | Conversión RGB a gris, modelo de color sigmoid y mapeo HSV directo |
| `utils_data.py` | Carga de imágenes y etiquetas desde el dataset |
| `app.py` | Servidor web Flask: identificación y búsqueda |
| `analysis.py` | Análisis comparativo de parámetros y métricas |

En la práctica el flujo es bastante lineal: empezamos preparando los datos, después dejamos los algoritmos configurados con los mejores parámetros encontrados, y finalmente los usamos para predecir y buscar.

## 4. Análisis de parámetros

En este apartado repasamos los parámetros que tienen más peso en el resultado y vemos qué pasa cuando los movemos.

---

### 4.1 Valor de K en KNN

En KNN, el valor de K es básicamente el número de vecinos que el algoritmo consulta para decidir a qué clase pertenece una imagen.

- Si **K es muy pequeña** (K=1), el algoritmo se queda con la clase del vecino más próximo. Puede ser muy preciso si las features son buenas, pero sensible al ruido con features malas.
- Si **K es muy grande** (K=21), la votación se diluye y las clases con menos ejemplos quedan dominadas por las mayoritarias.

Por eso lo que hemos hecho es ir probando varios valores de K y medir la **accuracy** sobre el conjunto de test en cada caso.

#### ¿Por qué se usa la métrica Accuracy?

La accuracy (exactitud) mide el porcentaje de imágenes clasificadas correctamente sobre el total:

$$\text{Accuracy} = \frac{\text{Predicciones correctas}}{\text{Total de muestras}}$$

Se escoge accuracy como métrica principal por las siguientes razones:

- El dataset está **relativamente equilibrado** entre clases (Shirts, Jeans, Dresses, etc.), por lo que accuracy no está sesgada hacia ninguna mayoritaria.
- El objetivo del sistema es clasificar correctamente el tipo de prenda. Un error en Sandals es **igual de importante** que un error en Shirts.
- Otras métricas como Precision o Recall tienen sentido cuando los costes de falso positivo y falso negativo son muy distintos (p.ej. diagnóstico médico). Aquí el coste es simétrico.
- F1-score es útil con desbalanceo severo. En este dataset la diferencia entre clases no justifica esa complejidad adicional.

Resumiendo: para un problema de clasificación multiclase con clases equilibradas, **accuracy es la métrica más directa e interpretable**.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# ── Datos experimentales ─────────────────────────────────────────────────
k_feat   = [1,      3,      5,      7,      9,      11]
acc_feat = [0.9109, 0.9056, 0.8949, 0.8870, 0.8936, 0.8856]

k_gray   = [1,      3,      5,      7,      9,      11,    15,    21]
acc_gray = [0.6310, 0.6875, 0.7202, 0.7083, 0.6964, 0.6845, 0.6607, 0.6250]

# ── Figura 1: Shape features vs Píxeles en gris ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Gráfico izquierda: Shape features
ax = axes[0]
ax.plot(k_feat, [a*100 for a in acc_feat],
        marker='o', lw=2, color='#4F46E5', ms=7,
        markerfacecolor='white', markeredgewidth=2)
ax.scatter([1], [acc_feat[0]*100], zorder=5, s=140, color='#4F46E5',
           label=f'Mejor: K=1  ({acc_feat[0]*100:.2f}%)')
ax.set_xlabel('Valor de K', fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('KNN con Shape Features', fontsize=12, fontweight='bold')
ax.set_xticks(k_feat)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax.set_ylim(86, 94)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Gráfico derecha: Comparativa
ax2 = axes[1]
ax2.plot(k_gray, [a*100 for a in acc_gray],
         marker='s', lw=2, color='#9CA3AF', ms=5, linestyle='--',
         markerfacecolor='white', markeredgewidth=2, label='Píxeles gris (original)')
ax2.plot(k_feat, [a*100 for a in acc_feat],
         marker='o', lw=2, color='#4F46E5', ms=6,
         markerfacecolor='white', markeredgewidth=2, label='Shape features (actual)')
ax2.scatter([1],  [acc_feat[0]*100], zorder=5, s=100, color='#4F46E5')
ax2.scatter([5],  [72.02],           zorder=5, s=80,  color='#9CA3AF', marker='D')
ax2.set_xlabel('Valor de K', fontsize=11)
ax2.set_ylabel('Accuracy (%)', fontsize=11)
ax2.set_title('Shape Features vs Píxeles en gris', fontsize=12, fontweight='bold')
ax2.set_xticks(sorted(set(k_gray + k_feat)))
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax2.set_ylim(55, 95)
ax2.grid(axis='y', linestyle='--', alpha=0.4)
ax2.legend(fontsize=9)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig.suptitle('Figura 1. Accuracy del KNN según el valor de K', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Tabla de resultados
print(f"{'K':>4} | {'Shape features':>16} | {'Píxeles gris':>13} | {'Diferencia':>11}")
print('-' * 55)
gray_dict = dict(zip(k_gray, acc_gray))
for k, af in zip(k_feat, acc_feat):
    ag = gray_dict.get(k)
    diff = f'+{(af-ag)*100:.2f}pp' if ag else '—'
    mark = ' <<<' if k == 1 else ''
    print(f"{k:>4} | {af*100:>15.2f}% | {ag*100 if ag else 0:>12.2f}% | {diff:>11}{mark}")

**Conclusión:** Con shape features, **K=1 obtiene la mayor accuracy (91.09%)**. El descriptor geométrico (aspect ratio ×8) ya distingue perfectamente las formas, por lo que votar entre varios vecinos solo añade ruido. Con píxeles en gris el mejor era K=5 con 72.02% — una mejora de **+19.07 puntos porcentuales**.

---

### 4.2 Número de colores en K-Means

En K-Means, la K significa cuántos grupos de color queremos sacar de la imagen. Como no todas las imágenes tienen el mismo número de colores importantes, usamos `find_bestK` para elegirlo automáticamente.

La función `find_bestK` prueba valores de K crecientes y mira cuánto baja la distancia intra-clase (WCD) en cada paso. Mientras la mejora siga siendo grande, seguimos subiendo K; en cuanto la **mejora relativa cae por debajo del 20%**, asume que añadir más grupos ya no compensa y se detiene.

$$\text{Criterio de parada:} \quad 100 - 100 \cdot \frac{WCD_k}{WCD_{k-1}} < 20\%$$

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# ── Datos WCD (threshold 20%) ────────────────────────────────────────────
ks   = [2,     3,     4,     5]
wcds = [18.43, 13.97, 11.68, 10.30]
imps = [0,
        (18.43 - 13.97) / 18.43 * 100,
        (13.97 - 11.68) / 13.97 * 100,
        (11.68 - 10.30) / 11.68 * 100]

fig, ax1 = plt.subplots(figsize=(7, 4))
ax2 = ax1.twinx()

ax1.plot(ks, wcds, marker='o', lw=2.5, color='#7C3AED', ms=8, label='WCD (distancia media)')
ax2.bar([k - 0.15 for k in ks], imps, width=0.3, alpha=0.35, color='#4F46E5', label='Mejora relativa (%)')
ax2.axhline(20, color='red', linestyle='--', lw=2, label='Umbral 20%')
ax2.scatter([4], [imps[2]], zorder=6, s=120, color='red',
            label=f'Parada: K=4 → {imps[2]:.1f}% < 20% → mejor K=3')

ax1.set_xlabel('K (número de clusters)', fontsize=11)
ax1.set_ylabel('WCD (distancia euclídea media)', fontsize=10, color='#7C3AED')
ax2.set_ylabel('Mejora relativa (%)', fontsize=10, color='#4F46E5')
ax1.set_title('Figura 2. Evolución del WCD al aumentar K (threshold 20%)',
              fontsize=12, fontweight='bold')
ax1.set_xticks(ks)
ax1.grid(axis='y', linestyle='--', alpha=0.3)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper right')
ax1.spines['top'].set_visible(False)
fig.tight_layout()
plt.show()

# Tabla WCD
print(f"\n{'K':>3} | {'WCD':>8} | {'Mejora':>8} | Decisión")
print('-' * 55)
for i, (k, wcd, imp) in enumerate(zip(ks, wcds, imps)):
    if i == 0:
        dec = 'Base'
    elif imp >= 20:
        dec = f'{imp:.1f}% >= 20% → continuar'
    else:
        dec = f'{imp:.1f}% < 20%  → PARAR → mejor K={ks[i-1]}'
    imp_str = f'{imp:.1f}%' if i > 0 else '—'
    print(f"{k:>3} | {wcd:>8.2f} | {imp_str:>8} | {dec}")

**Conclusión:** La gráfica muestra la lógica del criterio: el WCD cae con fuerza de K=2 a K=3 (24.2% ≥ 20%, continuamos), pero al pasar a K=4 la mejora baja al 16.4% < 20%, así que `find_bestK` se detiene y elige **K=3** como valor óptimo.

- En `Kmeans.py`, `find_bestK` usa un threshold del 20%, lo que lleva al mejor K=3.
- En `app.py` se usa `find_bestK(max_K=5)` con inicialización kmeans++ y `garment_mask`.

---

### 4.3 Inicialización de centroides

K-Means es bastante sensible a los centroides iniciales: si arrancan mal, el resultado final puede ser peor o el algoritmo puede tardar más en converger. Se han comparado tres métodos sobre 30 imágenes de test con `garment_mask`:

| Método | WCD medio | Iters medio | Observación |
|---|---|---|---|
| `first` | 18.52 | 8.1 | Primeros K píxeles — rápido pero sesgado |
| `random` | 17.94 | 11.8 | Píxeles aleatorios — variable entre ejecuciones |
| **`custom` (kmeans++)** | **16.81** | **9.3** | **ELEGIDO: mejor WCD y reproducible** |

**kmeans++** selecciona los centroides iniciales maximizando la distancia entre ellos, lo que asegura una buena cobertura del espacio de colores desde el inicio. Con `random_state=42` los resultados son reproducibles.

---

### 4.4 Características usadas en KNN (shape features)

Al principio simplemente le pasábamos a KNN la imagen en escala de grises aplanada, un vector de 4800 valores tal cual. La mejora más importante ha sido sustituirlo por un **descriptor de silueta compacto** implementado en `shape_features.py`:

| Componente | Pesos | Dimensiones |
|---|---|---|
| Máscara de silueta 32×32 | ×2.0 | 1024 |
| Imagen en gris 32×32 (fondo=blanco) | ×0.30 | 1024 |
| Proyecciones filas + columnas | ×2.0 | 64 |
| Descriptores escalares (aspect ratio, fill ratio, bbox, etc.) | ×8.0 | 7 |

El **peso alto (×8) en los descriptores geométricos** es clave: el aspect ratio solo ya distingue Jeans (muy alto ~2.2) de Shorts (~1.0) y Sandals (~0.6). Con píxeles en gris esa información quedaba diluida en 4800 valores.

Este descriptor aguanta mucho mejor las variaciones de posición y de fondo, lo que se nota sobre todo en prendas pequeñas como Sandals o Flip Flops.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Datos inicialización centroides (K=3, 30 imágenes, garment_mask) ─────
methods  = ['first\n(primeros K\npíxeles)', 'random\n(píxeles\naleatorios)', 'kmeans++\n(ELEGIDO)']
wcds     = [18.52, 17.94, 16.81]
iters    = [8.1,   11.8,  9.3]
colors_b = ['#CBD5E1', '#94A3B8', '#4F46E5']
colors_l = ['#E2E8F0', '#CBD5E1', '#EEF2FF']

x = np.arange(len(methods))
width = 0.38

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax2 = ax1.twinx()

bars1 = ax1.bar(x - width/2, wcds, width, color=colors_b,
                label='WCD medio (menor = mejor)', zorder=3,
                edgecolor='white', linewidth=1.5)
bars2 = ax2.bar(x + width/2, iters, width, color=colors_l,
                label='Iteraciones medias', zorder=3,
                edgecolor=colors_b, linewidth=1.5)

# Etiquetas sobre las barras
for bar, val in zip(bars1, wcds):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
             f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
for bar, val in zip(bars2, iters):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{val:.1f}', ha='center', va='bottom', fontsize=10)

ax1.set_ylabel('WCD medio (distancia euclídea)', fontsize=11)
ax2.set_ylabel('Iteraciones medias hasta convergencia', fontsize=11)
ax1.set_title('Figura 3. Comparativa de métodos de inicialización K-Means\n'
              '(K=3, 30 imágenes de test, garment_mask)',
              fontsize=12, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(methods, fontsize=10)
ax1.set_ylim(0, 22)
ax2.set_ylim(0, 16)
ax1.grid(axis='y', linestyle='--', alpha=0.4, zorder=0)
ax1.set_axisbelow(True)

ax1.get_xticklabels()[2].set_color('#4F46E5')
ax1.get_xticklabels()[2].set_fontweight('bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=10, loc='upper right')
ax1.spines['top'].set_visible(False)
ax2.spines['top'].set_visible(False)
fig.tight_layout()
plt.show()

print('\nResumen:')
print(f'{"Método":<12} {"WCD":<8} {"Iters":<8} {"Elección"}')
print('-' * 48)
for m, w, it in zip(["first", "random", "kmeans++"], wcds, iters):
    mark = ' <- ELEGIDO' if m == 'kmeans++' else ''
    print(f'{m:<12} {w:<8.2f} {it:<8.1f}{mark}')
print('\nkmeans++ obtiene el menor WCD con pocas iteraciones y resultados reproducibles.')

## 5. Análisis de eficiencia

El análisis de eficiencia nos sirve para hacernos una idea realista de cuánto tarda el sistema y qué recursos consume.

### 5.1 Eficiencia de KNN

KNN tiene una particularidad muy cómoda: prácticamente no se entrena. Lo único que hace en esa fase es guardar las imágenes de entrenamiento.

- **Entrenamiento:** O(N·D) solo para precalcular normas — prácticamente instantáneo.
- **Predicción:** O(N·D) por muestra, donde N = imágenes train, D = dimensión feature (~2199).
- **Memoria:** proporcional al tamaño del dataset de entrenamiento.

Con K=1 y shape features (D≈2199, N≈1600 train), la predicción de las imágenes de test tarda < 1s.

### 5.2 Eficiencia de K-Means

K-Means opera directamente sobre los píxeles de la prenda (tras `garment_mask`).

- **Coste por imagen:** O(N·K·I), con N píxeles de prenda (< 4800), K grupos e I iteraciones.
- Con kmeans++ y `garment_mask` el número de iteraciones es reducido (~9).

### 5.3 Eficiencia de find_bestK

`find_bestK` lanza K-Means varias veces seguidas, una por cada valor de K que prueba. El coste extra existe, pero es asumible porque el máximo de K es pequeño (≤ 5) y trabajamos sobre píxeles de prenda, no la imagen completa.

### 5.4 Resumen de eficiencia

| Componente | Ventaja | Limitación |
|---|---|---|
| KNN + shape features | Sin entrenamiento complejo. 91.09% acc con K=1. | Predicción O(N·D) — lineal con el dataset. |
| K-Means + garment_mask | Colores precisos al ignorar fondo y piel. | find_bestK ejecuta K-Means varias veces. |
| Precomputación | Búsqueda instantánea tras el arranque. | Arranque tarda ~2 min con 3.472 imágenes. |

## 6. Resultados globales del sistema

Más allá del ajuste de parámetros, esta sección resume el rendimiento del **sistema completo** sobre el conjunto de test: cuánto acierta al identificar el tipo de prenda (forma), cuánto al detectar el color, y cuánto cuando se le exige acertar **ambas etiquetas a la vez**.

| Métrica | Precisión |
|---|---|
| Precisión de forma (tipo de prenda) | 87.78% |
| Precisión de color | 82.00% |
| **Precisión conjunta (forma + color)** | **70.00%** |

El sistema identifica la **forma** con bastante fiabilidad (87.78%) y el **color** algo por debajo (82.00%). La **precisión conjunta** cae al 70.00% porque exige acertar las dos etiquetas en la misma imagen: cada error de forma o de color se descuenta del total, así que esta cifra es siempre menor o igual que las dos anteriores y refleja el rendimiento real de cara a una búsqueda combinada como "pink dress".


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Precisión global del sistema (conjunto de test) ──────────────────────
metrics = ['Forma\n(tipo de prenda)', 'Color', 'Conjunta\n(forma + color)']
values  = [87.78, 82.00, 70.00]
colors  = ['#94A3B8', '#94A3B8', '#4F46E5']  # se resalta la métrica conjunta

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(metrics, values, color=colors, width=0.6, zorder=3)

# Etiquetas de valor encima de cada barra
for bar, v in zip(bars, values):
    ax.annotate(f'{v:.2f}%', xy=(bar.get_x() + bar.get_width() / 2, v),
                xytext=(0, 4), textcoords='offset points',
                ha='center', va='bottom', fontweight='bold')

ax.set_ylim(0, 100)
ax.set_ylabel('Precisión (%)')
ax.set_title('Precisión global del sistema de etiquetado (test)', fontweight='bold')
ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
ax.set_axisbelow(True)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()


## 7. Conclusiones

El proyecto deja claro que no hace falta recurrir a algoritmos sofisticados para montar un sistema de etiquetado de imágenes útil. Con KNN y K-Means bien configurados se consiguen resultados más que razonables.

- La sustitución de píxeles en gris por **shape features** mejoró la accuracy del KNN de 72.02% (k=5) a **91.09% (k=1)**, una ganancia de **+19.07 puntos porcentuales**.
- **K=1 resulta óptimo** con shape features porque el descriptor geométrico (aspect ratio ×8) ya distingue perfectamente las formas, haciendo el voto entre vecinos contraproducente.
- Se eligió **accuracy** como métrica porque el dataset está equilibrado entre las 8 clases y el coste de error es simétrico.
- **kmeans++** con umbral 20% da mejor detección de colores que K fijo, adaptándose a prendas unicolor y multicolor.
- El número de colores no se fija a mano: **find_bestK** se queda con **K=3** por su cuenta, parando en cuanto la mejora del WCD baja del 20%.
- **garment_mask** (elimina fondo blanco y piel) mejora tanto la detección de forma como la de color.
- La **K** es clave en ambos algoritmos, aunque significa cosas distintas: en KNN son los vecinos que se consultan y en K-Means los grupos de color que se extraen.
- El buscador usa todas las 3.472 imágenes (train + test) aunque solo ~1.600 tienen etiqueta ground truth para el KNN.